In [1]:
import torch as th
import sys
import os
import matplotlib.pyplot as plt
import importlib.util
import numpy as np
import sys
import os


In [2]:
lpe_path = os.path.abspath(os.path.join('..', 'lpe'))

if lpe_path not in sys.path:
    sys.path.insert(0, lpe_path)

from lpe.method_utils import *
from lpe.utils import Transformer
from lpe.utils import datasets as lpe_datasets


In [3]:
## CONFIG

model_name = "gelu-4l"
device = th.device("cuda" if th.cuda.is_available() else "cpu")
model = Transformer.from_pretrained(model_name).to(device)

# initializing input distributions
dist_name = "camel"

gt_freqs = load_ground_truth(model_name, [dist_name], device=device)[dist_name] # ground truth tensor
gt_probs = gt_freqs / gt_freqs.sum()

lpe_datasets.USE_CACHE = True

In [19]:
num_tokens = 16
rand_tokens = pick_random_tokens(gt_freqs, num_tokens, 1e-9, 1e-5)
target_input = th.tensor([rand_tokens[0]])
print(f"target_input: {target_input}")
rand_inputs = th.tensor(rand_tokens[1:]).unsqueeze(-1)
print(f"target_input: {rand_inputs}")

target_input: tensor([29841])
target_input: tensor([[15414],
        [17942],
        [36317],
        [ 6152],
        [ 3012],
        [18302],
        [ 3410],
        [  925],
        [ 2988],
        [15442],
        [30192],
        [ 2265],
        [30787],
        [ 1054],
        [36121]])


In [20]:
dmodel = model.embed.d_model
activations_sum = th.zeros([len(model.blocks)+1, dmodel])

for input in rand_inputs:
    onehot = th.nn.functional.one_hot(input, num_classes=model.embed.d_vocab).float().to(device)
    onehot.requires_grad_(True)
    x = onehot @ model.embed.W_E

    x = x + model.pos_embed(input)
    activations_sum[0] += x.squeeze()
    for i, block in enumerate(model.blocks):
        x = block(x)
        activations_sum[i+1] += x.squeeze()

nontarget_acts = activations_sum / (num_tokens-1)
# print(nontarget_acts)


target_acts = th.zeros([len(model.blocks)+1, dmodel])
onehot = th.nn.functional.one_hot(target_input, num_classes=model.embed.d_vocab).float().to(device)
onehot.requires_grad_(True)
x = onehot @ model.embed.W_E

x = x + model.pos_embed(target_input)
target_acts[0] += x.squeeze()
for i, block in enumerate(model.blocks):
    x = block(x)
    target_acts[i+1] += x.squeeze()


x = model.ln_final(x)
logits_pre = model.unembed(x)
y = logits_pre.argmax(-1)

# print(target_acts)
print(f"target: {y}")
rand_input = th.tensor(pick_random_tokens(gt_freqs, 1, 1e-9, 1e-5))
print(rand_input)


target: tensor([[29841]])
tensor([6784])


In [ ]:
r_list = target_acts - nontarget_acts
print(r_list)

onehot = th.nn.functional.one_hot(rand_input, num_classes=model.embed.d_vocab).float().to(device)
onehot.requires_grad_(True)
x = onehot @ model.embed.W_E

ind = 2

x = x + model.pos_embed(rand_input)
for i, block in enumerate(model.blocks):
    x = block(x)
    # print("dosdsdfsd")
    # if i + 1 == ind:# or i+1 == ind + 1:
        # x = x + r_list[i+1]
    
x = model.ln_final(x)
logits_pre = model.unembed(x)
y = logits_pre.argmax(-1)

print(y)

tensor([[ -0.0825,  -0.2379,   0.2259,  ...,   0.2025,   0.1856,  -0.1045],
        [ -0.4714,  -0.8586,   1.3329,  ...,  -0.3822,  -0.0742,  -0.0612],
        [ -0.3832,   0.1279,   1.6465,  ...,  -0.8624,  -1.4779,  -1.1633],
        [ -0.1647,  -0.6630,   2.7271,  ...,  -0.1839,  -1.9000,  -1.7817],
        [  0.8798,   5.6764, -11.4866,  ...,   6.5422,  -2.3865,  -5.0472]],
       grad_fn=<SubBackward0>)
tensor([[16]])
